[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/20_weight_init_interview.ipynb)

# 🟢 Solution: Kaiming Initialization（面试版）

## 📋 题目描述

**难度：** Easy

实现 Kaiming（He）权重初始化。

Kaiming 初始化根据 fan-in 设置权重方差，以在 ReLU 网络中保持信号幅度，防止激活值消失或爆炸。

**签名:** `kaiming_init(weight) -> Tensor`

**参数:**
- `weight` — 需要原地初始化的张量 (out_features, in_features)

**返回:** 同一张量（原地操作）

**约束:**
- `std = sqrt(2 / fan_in)`，其中 `fan_in = weight.shape[1]`
- 用 `normal(0, std)` 填充
- 更小的 fan_in 应产生更大的 std

**提示：** `fan_in = weight.shape[1]` → `std = sqrt(2 / fan_in)`
`weight.normal_(0, std)` 原地操作 → 返回 `weight`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

import math

def kaiming_init(weight):
    # ---- Step 1: 获取 fan-in ----
    # fan_in = 输入维度数（权重矩阵的列数）
    fan_in = weight.shape[1] if weight.dim() >= 2 else weight.shape[0]

    # ---- Step 2: Kaiming 标准差 ----
    # 推导（面试要点）：
    # 假设 ReLU 网络，输入方差为 1
    # y = ReLU(Σ w_i * x_i)
    # E[y²] = fan_in * Var(w) * E[ReLU(x)²]
    # 由于 ReLU 丢弃一半：E[ReLU(x)²] = 1/2
    # 要保持 E[y²] = 1：
    #   fan_in * Var(w) * 1/2 = 1
    #   Var(w) = 2 / fan_in
    #   std = sqrt(2 / fan_in)
    std = math.sqrt(2.0 / fan_in)

    # ---- Step 3: 原地填充 ----
    # .normal_() 是原地操作，直接修改传入的张量
    # 必须在 no_grad 下执行，否则 PyTorch 会追踪这个操作
    with torch.no_grad():
        weight.normal_(0, std)
    return weight

# 面试常见追问：
# Q: Kaiming vs Xavier 的区别？
# A: Xavier: std = sqrt(2 / (fan_in + fan_out))，适合 sigmoid/tanh
#    Kaiming: std = sqrt(2 / fan_in)，适合 ReLU
#    区别在于 Kaiming 考虑了 ReLU 丢弃一半神经元的影响
# Q: 为什么不用均匀分布？
# A: Kaiming 论文实验表明正态分布效果略好，但均匀分布版本也可以
# Q: 偏置怎么初始化？
# A: 通常初始化为 0。


In [ ]:
w = torch.empty(512, 256)
kaiming_init(w)
print(f'Shape: {w.shape}, Std: {w.std():.4f}, Expected: {math.sqrt(2/256):.4f}')


In [ ]:
from torch_judge import check
check('weight_init')


## 📝 核心思路总结

1. **Kaiming 公式**：`std = sqrt(2 / fan_in)`，为 ReLU 网络量身定制
2. **推导关键**：ReLU 丢弃一半神经元 → 方差减半 → 需要 `2/fan_in` 补偿
3. **原地操作**：`weight.normal_(0, std)` 在 `no_grad()` 下执行
4. **vs Xavier**：Xavier 用 `2/(fan_in+fan_out)`，适合无 ReLU 的网络
